In [4]:
with open('/Users/justlikethat/langgraph-course/resourses/artical_web_rag.txt','r') as file:
    text = " ".join(line.rstrip() for line in file)

# s = text
s = "dog, cat, roadrunner, bird, chiken, hourse, wale, chiken, wale, roadrunner".split()
print(sorted(list(set(s))))

['bird,', 'cat,', 'chiken,', 'dog,', 'hourse,', 'roadrunner', 'roadrunner,', 'wale,']


In [5]:
import random
import numpy as np

In [6]:
def rand_vec(dim):
    return np.random.uniform(-3.0, 3.0, size=dim)

def logsumexp(x):
    m = np.max(x)
    return m + np.log(np.sum(np.exp(x - m)))

def softmax(x):
    x = x - np.max(x)
    e = np.exp(x)
    return e / e.sum()

In [7]:
dim = 3
lr = 0.05
# d[word] = (v_w, u_w)  center vec, context 
d = {}
# s is whole corpus
for tok in s:
    if tok not in d:
        d[tok] = (rand_vec(dim), rand_vec(dim))
vocab = list(d.keys())
w2i = {w:i for i,w in enumerate(vocab)}
V = np.stack([d[w][0] for w in vocab], axis=0)  # center vectors (|V|, dim)
U = np.stack([d[w][1] for w in vocab], axis=0)  # context vectors (|V|, dim

In [12]:
for _ in range(1):
    for i in range(len(s) - 5 + 1): 
        win = s[i:i+5]
        print(win)
        c_word = win[2]
        c = w2i[c_word]
    
        for j, t_word in enumerate(win):
            if j == 2: 
                continue
            t = w2i[t_word]
            v_c = V[c]                 # (dim,)
            z = U @ v_c                # (|V|,)
            p = softmax(z)             # (|V|,)
    
            # g = p - y
            g = p.copy()
            g[t] -= 1.0                # (|V|,)
    
            # gradients
            grad_v = U.T @ g           # (dim,)
            grad_U = np.outer(g, v_c)  # (|V|, dim)
    
            # SGD step
            V[c] -= lr * grad_v
            U    -= lr * grad_U
    loss = 0.0
    pair_count = 0
    for i in range(len(s) - 5 + 1):
        win = s[i:i+5]
        center = win[2]
        v_center = d[center][0]     # v_w
        logits = U @ v_center       # (|V|,)  logits for all context words
        logZ = logsumexp(logits)    # log sum exp over vocab
        for j, ctx in enumerate(win):
            if j == 2:
                continue
            # log p(ctx | center) = (u_ctx·v_center) - logZ
            u_ctx = d[ctx][1]
            loss -= (u_ctx @ v_center) - logZ   # negative log-likelihood
            pair_count += 1
    
    avg_loss = loss / max(pair_count, 1)
    print("loss:", loss, "avg_loss:", avg_loss, "pairs:", pair_count)

['dog,', 'cat,', 'roadrunner,', 'bird,', 'chiken,']
[[ 0.12005794 -1.87877597  1.1102766 ]
 [ 1.21894569 -1.48421961  2.06753311]
 [ 0.42022599  0.63114848 -2.81731766]
 [ 0.67123986 -1.59776686  1.89775137]
 [-0.72480693 -0.93715697 -0.68707254]
 [-2.15622481 -2.8315075  -0.69760764]
 [-1.71602718 -0.0273711   0.29716174]
 [-1.6662984  -1.85338746 -2.11107416]] UUUU
[ 4.3056085   4.50774946 -8.89181979  5.16684586  0.27115347  4.46930147
  4.02956811 -1.05514487] zzzzz
[[ 0.04089259 -1.91347911  1.2244048 ]
 [ 1.23640787 -1.47656484  2.04235887]
 [ 0.42022601  0.63114849 -2.8173177 ]
 [ 0.70499504 -1.58296985  1.84908845]
 [-0.72455449 -0.93704631 -0.68743648]
 [-2.13942127 -2.82414146 -0.72183235]
 [-1.70520221 -0.02262583  0.28155598]
 [-1.66623139 -1.85335809 -2.11117077]] UUUU
[ 4.80875242  4.42725233 -8.88895022  4.98243208  0.27321992  4.37548666
  3.94428684 -1.04960798] zzzzz
[[ 0.06437198 -1.90298569  1.1903777 ]
 [ 1.15956177 -1.51090894  2.15372675]
 [ 0.42022604  0.6311485

In [16]:
def rand_vec(dim):
    return np.random.uniform(-3.0, 3.0, size=dim)

def logsumexp(x):
    m = np.max(x)
    return m + np.log(np.sum(np.exp(x - m)))

def softmax(x):
    x = x - np.max(x)
    e = np.exp(x)
    return e / e.sum()
dim = 3
lr = 0.05
# d[word] = (v_w, u_w)  center vec, context 
d = {}
# s is whole corpus
for tok in s:
    if tok not in d:
        d[tok] = (rand_vec(dim), rand_vec(dim))
vocab = list(d.keys())
w2i = {w:i for i,w in enumerate(vocab)}
V = np.stack([d[w][0] for w in vocab], axis=0)  # center vectors (|V|, dim)
U = np.stack([d[w][1] for w in vocab], axis=0)  # context vectors (|V|, dim
for _ in range(10000):
    for i in range(len(s) - 5 + 1): 
        win = s[i:i+5]
        c_word = win[2]
        c = w2i[c_word]
    
        for j, t_word in enumerate(win):
            if j == 2: 
                continue
            t = w2i[t_word]
    
            v_c = V[c]                 # (dim,)
            z = U @ v_c                # (|V|,)
            p = softmax(z)             # (|V|,)
    
            # g = p - y
            g = p.copy()
            g[t] -= 1.0                # (|V|,)
    
            # gradients
            grad_v = U.T @ g           # (dim,)
            grad_U = np.outer(g, v_c)  # (|V|, dim)
    
            # SGD step
            V[c] -= lr * grad_v
            U    -= lr * grad_U
    loss = 0.0
    pair_count = 0
    for i in range(len(s) - 5 + 1):
        win = s[i:i+5]
        center = win[2]
        v_center = V[w2i[center]]
        logits = U @ v_center       # (|V|,)  logits for all context words
        logZ = logsumexp(logits)    # log sum exp over vocab
        for j, ctx in enumerate(win):
            if j == 2:
                continue
            # log p(ctx | center) = (u_ctx·v_center) - logZ
            u_ctx = U[w2i[ctx]]
            loss -= (u_ctx @ v_center) - logZ   # negative log-likelihood
            pair_count += 1
    
    avg_loss = loss / max(pair_count, 1)
print("loss:", loss, "avg_loss:", avg_loss, "pairs:", pair_count)

loss: 32.0608729670894 avg_loss: 1.3358697069620584 pairs: 24


In [14]:
for _ in range(100):
    for i in range(len(s) - 5 + 1): 
        win = s[i:i+5]
        c_word = win[2]
        c = w2i[c_word]
    
        for j, t_word in enumerate(win):
            if j == 2: 
                continue
            t = w2i[t_word]
    
            v_c = V[c]                 # (dim,)
            z = U @ v_c                # (|V|,)
            p = softmax(z)             # (|V|,)
    
            # g = p - y
            g = p.copy()
            g[t] -= 1.0                # (|V|,)
    
            # gradients
            grad_v = U.T @ g           # (dim,)
            grad_U = np.outer(g, v_c)  # (|V|, dim)
    
            # SGD step
            V[c] -= lr * grad_v
            U    -= lr * grad_U
    loss = 0.0
    pair_count = 0
    for i in range(len(s) - 5 + 1):
        win = s[i:i+5]
        center = win[2]
        v_center = d[center][0]     # v_w
        logits = U @ v_center       # (|V|,)  logits for all context words
        logZ = logsumexp(logits)    # log sum exp over vocab
        for j, ctx in enumerate(win):
            if j == 2:
                continue
            # log p(ctx | center) = (u_ctx·v_center) - logZ
            u_ctx = d[ctx][1]
            loss -= (u_ctx @ v_center) - logZ   # negative log-likelihood
            pair_count += 1
    
    avg_loss = loss / max(pair_count, 1)
    print("loss:", loss, "avg_loss:", avg_loss, "pairs:", pair_count)

loss: 93.60814102463725 avg_loss: 3.900339209359885 pairs: 24
loss: 90.73441542028829 avg_loss: 3.7806006425120118 pairs: 24
loss: 89.0528104225459 avg_loss: 3.7105337676060794 pairs: 24
loss: 87.94160712072579 avg_loss: 3.664233630030241 pairs: 24
loss: 87.15957597446265 avg_loss: 3.631648998935944 pairs: 24
loss: 86.59583219961982 avg_loss: 3.6081596749841593 pairs: 24
loss: 86.19388318041949 avg_loss: 3.5914117991841454 pairs: 24
loss: 85.92261735760728 avg_loss: 3.58010905656697 pairs: 24
loss: 85.76293733508169 avg_loss: 3.5734557222950705 pairs: 24
loss: 85.70055139491728 avg_loss: 3.5708563081215536 pairs: 24
loss: 85.7217757005362 avg_loss: 3.5717406541890084 pairs: 24
loss: 85.81139745939052 avg_loss: 3.5754748941412715 pairs: 24
loss: 85.95234052810254 avg_loss: 3.5813475220042723 pairs: 24
loss: 86.12679199581466 avg_loss: 3.588616333158944 pairs: 24
loss: 86.31806168012348 avg_loss: 3.5965859033384784 pairs: 24
loss: 86.51226909753316 avg_loss: 3.604677879063882 pairs: 24
l

In [27]:
def build_training_pairs(tokens: list[str], window_size: int) -> list[tuple[str, str]]:
    """Create (center, context) pairs from sliding windows."""
    if window_size % 2 == 0:
        raise ValueError("window_size must be odd")

    half = window_size // 2
    pairs: list[tuple[str, str]] = []
    for i in range(half, len(tokens) - half):
        center = tokens[i]
        for j in range(i - half, i + half + 1):
            if j == i:
                continue
            pairs.append((center, tokens[j]))
    return pairs

In [28]:
s = "dog, cat, roadrunner, bird, chiken, hourse, wale, chiken, wale, roadrunner".split()

In [29]:
print(build_training_pairs(s, 5))

[('roadrunner,', 'dog,'), ('roadrunner,', 'cat,'), ('roadrunner,', 'bird,'), ('roadrunner,', 'chiken,'), ('bird,', 'cat,'), ('bird,', 'roadrunner,'), ('bird,', 'chiken,'), ('bird,', 'hourse,'), ('chiken,', 'roadrunner,'), ('chiken,', 'bird,'), ('chiken,', 'hourse,'), ('chiken,', 'wale,'), ('hourse,', 'bird,'), ('hourse,', 'chiken,'), ('hourse,', 'wale,'), ('hourse,', 'chiken,'), ('wale,', 'chiken,'), ('wale,', 'hourse,'), ('wale,', 'chiken,'), ('wale,', 'wale,'), ('chiken,', 'hourse,'), ('chiken,', 'wale,'), ('chiken,', 'wale,'), ('chiken,', 'roadrunner')]


In [125]:
h = 0.001
a = 2.0
b = -3.0
c = 10.0
f = -2.0

In [126]:
d1 = ((a * b) + c) * f
d2 = (((a * b) + c)) * (f+h)
(d2 - d1)/h, d1, 'y'

(3.9999999999995595, -8.0, 'y')

In [127]:
d1 = ((a * b) + c) * f
d2 = ((((a * b) + c)+h) * (f))
(d2 - d1)/h, ((a * b) + c), 'n'

(-2.000000000000668, 4.0, 'n')

In [128]:
d1 = ((a * b) + c) * f
d2 = ((((a * b) + (c+h))) * (f))
(d2 - d1)/h, c

(-1.9999999999988916, 10.0)

In [129]:
d1 = ((a * b) + c) * f
d2 = ( ( (((a * b)+h) + (c)) ) * (f) )
(d2 - d1)/h, (a * b), 'm'

(-2.000000000000668, -6.0, 'm')

In [130]:
d1 = (((a) * b) + c) * f
d2 = ( ( ((((a+h) * b)) + (c)) ) * (f) )
(d2 - d1)/h, a

(6.000000000000227, 2.0)

In [131]:
d1 = ((a * (b)) + c) * f
d2 = ( ( (((a * (b+h))) + (c)) ) * (f) )
(d2 - d1)/h, b

(-3.9999999999995595, -3.0)

In [141]:
a = [1, 2, 3]

In [142]:
[i + 1 for i in a]

[2, 3, 4]

In [148]:
b = (i + 1 for i in a)